In [7]:


def agent(query: str):
    query_lower = query.lower()

    try:
        if "calculate" in query_lower:
            expression = query_lower.replace("calculate", "").strip()
            result = calculator(expression)

            if result == "Error in calculation":
                return {"type": "error", "result": result}

            return {"type": "calculation", "result": result}

        elif "keywords" in query_lower:
            keywords = extract_keywords(query)
            return {"type": "keywords", "result": keywords}

        else:
            return {"type": "general", "result": f"I don't have a specific tool for this, but you asked: '{query}'"}

    except Exception as e:
        return {"type": "error", "result": f"Agent error: {str(e)}"}

In [8]:
queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?"
]
for q in queries:
    print("Query:", q)
    print("Response:", agent(q))
    print("-" * 50)

Query: Calculate 20 + 5
Response: {'type': 'error', 'result': "Agent error: name 'calculator' is not defined"}
--------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
Response: {'type': 'error', 'result': "Agent error: name 'extract_keywords' is not defined"}
--------------------------------------------------
Query: What is machine learning?
Response: {'type': 'general', 'result': "I don't have a specific tool for this, but you asked: 'What is machine learning?'"}
--------------------------------------------------


In [9]:
import re
import logging
from datetime import datetime

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("agent")

def word_stats(text: str) -> dict:
    """Return basic stats about a piece of text."""
    try:
        words = re.findall(r"\b\w+\b", text)
        return {"word_count": len(words), "char_count": len(text)}
    except Exception:
        return {"error": "Could not compute stats"}


def calculator_safe(expression: str) -> str:
    """Safer calculator: only allow digits, operators, parentheses, decimal points."""
    try:
        if not re.fullmatch(r"[0-9\.\+\-\*\/\(\)\s]+", expression):
            return "Error in calculation"
        return str(eval(expression))
    except Exception:
        return "Error in calculation"


def extract_keywords_v2(text: str, top_n: int = 5) -> list:
    """Extract keywords, stripping punctuation and common stopwords."""
    stopwords = {"the", "is", "are", "and", "from", "this", "that", "with", "into"}
    try:
        words = re.findall(r"\b[a-zA-Z]{5,}\b", text)  # words with 5+ letters
        keywords = [w.lower() for w in words if w.lower() not in stopwords]
        seen = []
        for w in keywords:
            if w not in seen:
                seen.append(w)
        return seen[:top_n]
    except Exception:
        return []


def agent_v2(query: str):
    query_lower = query.lower()
    logger.info(f"Received query: {query}")

    try:
        if re.search(r"\bcalculate\b", query_lower) or re.search(r"[\d]+\s*[\+\-\*/]\s*[\d]+", query_lower):
            expression = re.sub(r"calculate", "", query_lower).strip()
            result = calculator_safe(expression)
            response = (
                {"type": "error", "result": result}
                if result == "Error in calculation"
                else {"type": "calculation", "result": result}
            )

        elif re.search(r"\bkeyword", query_lower):
            keywords = extract_keywords_v2(query)
            response = {"type": "keywords", "result": keywords}

        elif re.search(r"\bstats\b|\bword count\b", query_lower):
            response = {"type": "stats", "result": word_stats(query)}

        else:
            response = {"type": "general", "result": f"I don't have a specific tool for this, but you asked: '{query}'"}

    except Exception as e:
        logger.error(f"Agent failed: {e}")
        response = {"type": "error", "result": f"Agent error: {str(e)}"}

    logger.info(f"Response: {response}")
    return response

In [10]:


while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    
    if user_input.lower() == "exit":
        print("Exit!")
        break
    
    if not user_input.strip():
        print("Please enter a query.")
        continue

    response = agent_v2(user_input)
    print("Response:", response)
    print("-" * 50)

Enter query (type 'exit' to stop):  calculate 2+10


2026-07-16 14:58:32,268 | INFO | Received query: calculate 2+10
2026-07-16 14:58:32,270 | INFO | Response: {'type': 'calculation', 'result': '12'}


Response: {'type': 'calculation', 'result': '12'}
--------------------------------------------------


Enter query (type 'exit' to stop):  exit


Goodbye!
